
# Ensemble Demo: Bagging and Boosting 

This notebook compares **base classifiers** with **ensemble versions** on the **Breast Cancer Wisconsin Diagnostic** dataset, a real-world medical classification dataset included in scikit-learn.

## Studies
- Compare **base models** vs **bagging**
- Show **Random Forest** as a specialized bagged tree ensemble
- Compare **base models** vs **boosting**
- Summarize results with common classification metrics


In [1]:

# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone

# Base classifiers
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

# Ensembles
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier,
    AdaBoostClassifier
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [3]:

# Load the real-world dataset
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

print("Dataset:", data.DESCR.split("\n")[0])
print("Samples:", X.shape[0])
print("Features:", X.shape[1])
print("Classes:", list(data.target_names))

display(X.head())


Dataset: .. _breast_cancer_dataset:
Samples: 569
Features: 30
Classes: [np.str_('malignant'), np.str_('benign')]


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [4]:

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])
print("\nClass balance in training set:")
display(y_train.value_counts(normalize=True).rename("proportion"))


Training samples: 426
Test samples: 143

Class balance in training set:


target
1    0.626761
0    0.373239
Name: proportion, dtype: float64

## Helper functions

The next cell includes:
- a compatibility helper for `BaggingClassifier`
- a compatibility helper for `AdaBoostClassifier`
- a reusable evaluation function


In [5]:

def make_bagging(estimator, n_estimators=100, max_samples=0.8, max_features=1.0,
                 bootstrap=True, random_state=RANDOM_STATE):
    """Create a BaggingClassifier that works across scikit-learn versions."""
    try:
        return BaggingClassifier(
            estimator=estimator,
            n_estimators=n_estimators,
            max_samples=max_samples,
            max_features=max_features,
            bootstrap=bootstrap,
            random_state=random_state,
            n_jobs=-1
        )
    except TypeError:
        return BaggingClassifier(
            base_estimator=estimator,
            n_estimators=n_estimators,
            max_samples=max_samples,
            max_features=max_features,
            bootstrap=bootstrap,
            random_state=random_state,
            n_jobs=-1
        )

def make_adaboost(estimator, n_estimators=100, learning_rate=0.5, random_state=RANDOM_STATE):
    """Create an AdaBoostClassifier that works across scikit-learn versions."""
    try:
        return AdaBoostClassifier(
            estimator=estimator,
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            random_state=random_state
        )
    except TypeError:
        return AdaBoostClassifier(
            base_estimator=estimator,
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            random_state=random_state
        )

def get_scores(model, X_train, X_test, y_train, y_test, cv):
    model = clone(model)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results = {
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
    }

    # ROC-AUC when probability estimates are available
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
        results["test_roc_auc"] = roc_auc_score(y_test, y_score)
    else:
        results["test_roc_auc"] = np.nan

    cv_results = cross_validate(
        clone(model),
        X_train,
        y_train,
        cv=cv,
        scoring=["accuracy", "f1", "roc_auc"],
        n_jobs=-1
    )

    results["cv_accuracy_mean"] = cv_results["test_accuracy"].mean()
    results["cv_accuracy_std"] = cv_results["test_accuracy"].std()
    results["cv_f1_mean"] = cv_results["test_f1"].mean()
    results["cv_roc_auc_mean"] = cv_results["test_roc_auc"].mean()

    return results, model


## Models to compare

### Base classifiers
- Decision Tree
- Logistic Regression
- KNN
- Decision Stump (depth = 1)
- Shallow Tree (depth = 2)

### Bagging
- Bagging + Decision Tree
- Bagging + Logistic Regression
- Bagging + KNN
- Random Forest

### Boosting
- AdaBoost + Decision Stump
- AdaBoost + Shallow Tree


In [6]:

# Base models
base_models = {
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=7))
    ]),
    "Decision Stump (depth=1)": DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
    "Shallow Tree (depth=2)": DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE),
}

# Bagging models
bagging_models = {
    "Bagging + Decision Tree": make_bagging(
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        n_estimators=100,
        max_samples=0.8
    ),
    "Bagging + Logistic Regression": make_bagging(
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
        ]),
        n_estimators=50,
        max_samples=0.8
    ),
    "Bagging + KNN": make_bagging(
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=7))
        ]),
        n_estimators=50,
        max_samples=0.8
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
}

# Boosting models
boosting_models = {
    "AdaBoost + Decision Stump": make_adaboost(
        DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
        n_estimators=200,
        learning_rate=0.5
    ),
    "AdaBoost + Shallow Tree": make_adaboost(
        DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE),
        n_estimators=150,
        learning_rate=0.5
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


In [7]:

# Evaluate all models
all_models = {}
all_models.update(base_models)
all_models.update(bagging_models)
all_models.update(boosting_models)

rows = []
fitted_models = {}

for name, model in all_models.items():
    scores, fitted_model = get_scores(model, X_train, X_test, y_train, y_test, cv)
    fitted_models[name] = fitted_model
    row = {"model": name}
    row.update(scores)
    rows.append(row)

results_df = pd.DataFrame(rows).sort_values(
    by=["test_accuracy", "cv_accuracy_mean", "test_f1"],
    ascending=False
).reset_index(drop=True)

pd.set_option("display.precision", 4)
display(results_df)


,model,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,cv_accuracy_mean,cv_accuracy_std,cv_f1_mean,cv_roc_auc_mean
0,Logistic Regression,0.9860,0.9889,0.9889,0.9889,0.9977,0.9789,0.0088,0.9833,0.9962
1,Bagging + Logistic Regression,0.9790,0.9780,0.9889,0.9834,0.9973,0.9766,0.0104,0.9815,0.9960
2,KNN,0.9790,0.9677,1.0000,0.9836,0.9923,0.9648,0.0129,0.9726,0.9887
3,Bagging + KNN,0.9720,0.9574,1.0000,0.9783,0.9949,0.9695,0.0057,0.9761,0.9881
4,AdaBoost + Shallow Tree,0.9580,0.9565,0.9778,0.9670,0.9897,0.9789,0.0088,0.9833,0.9943
5,AdaBoost + Decision Stump,0.9580,0.9565,0.9778,0.9670,0.9878,0.9695,0.0141,0.9760,0.9931
6,Random Forest,0.9580,0.9565,0.9778,0.9670,0.9949,0.9578,0.0175,0.9665,0.9881
7,Bagging + Decision Tree,0.9580,0.9565,0.9778,0.9670,0.9942,0.9507,0.0156,0.9604,0.9883
8,Decision Tree,0.9231,0.9540,0.9222,0.9379,0.9234,0.9248,0.0161,0.9396,0.9224
9,Decision Stump (depth=1),0.9231,0.9158,0.9667,0.9405,0.9079,0.8920,0.0190,0.9171,0.8704



## A focused comparison: base model vs ensemble version

This makes the ensemble effect easier to discuss in class.


In [ ]:

comparison_pairs = [
    ("Decision Tree", "Bagging + Decision Tree"),
    ("Decision Tree", "Random Forest"),
    ("Logistic Regression", "Bagging + Logistic Regression"),
    ("KNN", "Bagging + KNN"),
    ("Decision Stump (depth=1)", "AdaBoost + Decision Stump"),
    ("Shallow Tree (depth=2)", "AdaBoost + Shallow Tree"),
]

comparison_rows = []
for base_name, ensemble_name in comparison_pairs:
    base_row = results_df.loc[results_df["model"] == base_name].iloc[0]
    ens_row = results_df.loc[results_df["model"] == ensemble_name].iloc[0]

    comparison_rows.append({
        "base_model": base_name,
        "ensemble_model": ensemble_name,
        "base_test_accuracy": base_row["test_accuracy"],
        "ensemble_test_accuracy": ens_row["test_accuracy"],
        "accuracy_gain": ens_row["test_accuracy"] - base_row["test_accuracy"],
        "base_test_f1": base_row["test_f1"],
        "ensemble_test_f1": ens_row["test_f1"],
        "f1_gain": ens_row["test_f1"] - base_row["test_f1"],
    })

pair_df = pd.DataFrame(comparison_rows).sort_values(by="accuracy_gain", ascending=False)
display(pair_df)


In [ ]:

# Plot test accuracy for all models
plot_df = results_df.sort_values("test_accuracy", ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(plot_df["model"], plot_df["test_accuracy"])
plt.xlabel("Test Accuracy")
plt.title("Model Comparison: Test Accuracy")
plt.xlim(max(0.85, plot_df["test_accuracy"].min() - 0.02), 1.0)
plt.tight_layout()
plt.show()


In [ ]:

# Plot cross-validation accuracy for all models
plot_df = results_df.sort_values("cv_accuracy_mean", ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(plot_df["model"], plot_df["cv_accuracy_mean"])
plt.xlabel("Mean CV Accuracy")
plt.title("Model Comparison: 5-Fold Cross-Validation Accuracy")
plt.xlim(max(0.85, plot_df["cv_accuracy_mean"].min() - 0.02), 1.0)
plt.tight_layout()
plt.show()


In [ ]:

# Confusion matrices for a few representative models
selected_models = [
    "Decision Tree",
    "Random Forest",
    "Decision Stump (depth=1)",
    "AdaBoost + Decision Stump"
]

for name in selected_models:
    model = fitted_models[name]
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=data.target_names)
    disp.plot()
    plt.title(name)
    plt.show()


## Discussion prompts for class

### Bagging
- Which base models benefit the most from bagging?
- Why do **high-variance models** often gain more from bagging?
- Why is Random Forest usually better than a single decision tree?

### Boosting
- How much does AdaBoost improve over a single stump?
- Does boosting help a stronger tree too?

### Bias-variance interpretation
- Bagging mainly reduces **variance**
- Boosting often reduces **bias** by sequentially correcting errors


In [ ]:
# Optional: inspect feature importance for Random Forest
rf = fitted_models["Random Forest"]

rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

print("Top 10 Random Forest feature importances")
display(rf_importance.to_frame("importance"))



## Extension ideas
- Tune `n_estimators`, `max_depth`, and `learning_rate`
- Add ROC curves for the best models
- Repeat on another real dataset such as Wine or a clinical dataset you already use in class
- Use this notebook as a lecture demo, then let students modify one bagging model and one boosting model
